# IndPenSim Dataset — Overview & EDA

**Purpose**: Exploratory data analysis of 100-batch industrial penicillin fermentation simulation data  
**Data**: `100_Batches_IndPenSim_V3.csv` (time-series), `100_Batches_IndPenSim_Statistics.csv` (batch summary)  

---
**Analysis Outline**
1. Library & Data Loading
2. Data Structure Overview
3. Batch Statistics (Fault ratio, Yield distribution)
4. Process Variable Time-Series Exploration
5. Raman Spectroscopy Data Exploration
6. Missing Value Analysis
7. Fault vs Normal Batch Comparison
8. Summary & Research Direction

## 1. Library & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

DATA_DIR   = Path('../data/raw')
STATS_PATH = DATA_DIR / '100_Batches_IndPenSim_Statistics.csv'
MAIN_PATH  = DATA_DIR / '100_Batches_IndPenSim_V3.csv'

BATCH_COL       = 'Batch reference(Batch_ref:Batch ref)'
BATCH_COL_STATS = 'Batch ref'
FAULT_COL       = 'Fault ref(0-NoFault 1-Fault)'
PAT_COL         = '2-PAT control(PAT_ref:PAT ref)'
TIME_COL        = 'Time (h)'
YIELD_COL       = 'Penicllin_yield_total (kg)'

In [ ]:
df_stats = pd.read_csv(STATS_PATH)
df_stats.columns = df_stats.columns.str.strip()
print('Statistics shape:', df_stats.shape)
df_stats.head()

In [ ]:
# Large file — may take tens of seconds
df = pd.read_csv(MAIN_PATH)
df.columns = df.columns.str.strip()
print('Main data shape:', df.shape)
df.head(3)

## 2. Data Structure Overview

In [ ]:
raman_cols   = [c for c in df.columns if c.strip().lstrip('-').isdigit()]
process_cols = [c for c in df.columns if c not in raman_cols]

print(f'Total columns        : {df.shape[1]}')
print(f'Process variables    : {len(process_cols)}')
print(f'Raman channels       : {len(raman_cols)}')
print(f'Wavenumber range     : {min(int(c) for c in raman_cols)} ~ {max(int(c) for c in raman_cols)} cm\u207b\u00b9')
print()
print('Process variable list:')
for i, c in enumerate(process_cols):
    print(f'  [{i:2d}] {c}')

In [ ]:
batch_lengths = df[BATCH_COL].value_counts().sort_index()

print(f'Number of batches    : {batch_lengths.shape[0]}')
print(f'Timesteps (min)      : {batch_lengths.min()}')
print(f'Timesteps (max)      : {batch_lengths.max()}')
print(f'Timesteps (mean)     : {batch_lengths.mean():.1f}')

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(batch_lengths.index, batch_lengths.values, color='steelblue', width=0.8)
ax.set_xlabel('Batch ID')
ax.set_ylabel('Timesteps')
ax.set_title('Timesteps per Batch')
plt.tight_layout()
plt.show()

## 3. Batch Statistics

In [ ]:
fault_counts = df_stats[FAULT_COL].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].pie(
    fault_counts.values,
    labels=['No-Fault', 'Fault'],
    autopct='%1.0f%%',
    colors=['#4C9BE8', '#E8644C'],
    startangle=90
)
axes[0].set_title('Fault Batch Ratio')

for label, color in zip([0, 1], ['#4C9BE8', '#E8644C']):
    subset = df_stats[df_stats[FAULT_COL] == label][YIELD_COL]
    axes[1].hist(subset, bins=15, alpha=0.7, color=color,
                 label='No-Fault' if label == 0 else 'Fault')
axes[1].set_xlabel('Penicillin Yield Total (kg)')
axes[1].set_ylabel('Count')
axes[1].set_title('Yield Distribution (Fault vs No-Fault)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df_stats.groupby(FAULT_COL)[YIELD_COL].describe().round(0))

## 4. Process Variable Time-Series Exploration

In [ ]:
key_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
    'Substrate concentration(S:g/L)',
    'Vessel Volume(V:L)',
]

fault_batch_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values[:1]
no_fault_batch_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values[:2]

fig, axes = plt.subplots(len(key_vars), 1, figsize=(13, 14), sharex=False)

for ax, var in zip(axes, key_vars):
    for bid in no_fault_batch_ids:
        sub = df[df[BATCH_COL] == bid]
        ax.plot(sub[TIME_COL], sub[var], color='#4C9BE8', alpha=0.8, linewidth=1.2, label='No-Fault')
    for bid in fault_batch_ids:
        sub = df[df[BATCH_COL] == bid]
        ax.plot(sub[TIME_COL], sub[var], color='#E8644C', alpha=0.9, linewidth=1.4, linestyle='--', label='Fault')
    ax.set_ylabel(var.split('(')[0].strip(), fontsize=9)
    ax.tick_params(axis='both', labelsize=8)

axes[0].legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('Time (h)')
fig.suptitle('Key Process Variables Time-Series (Sample Batches)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
var = 'Penicillin concentration(P:g/L)'
grouped  = df.groupby(TIME_COL)[var]
mean_val = grouped.mean()
std_val  = grouped.std()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mean_val.index, mean_val.values, color='steelblue', linewidth=1.5, label='Mean')
ax.fill_between(mean_val.index, mean_val - std_val, mean_val + std_val,
                alpha=0.25, color='steelblue', label='\u00b11 Std')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Penicillin (g/L)')
ax.set_title('Penicillin Concentration \u2014 All Batches Mean \u00b1 Std')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Raman Spectroscopy Data Exploration

In [ ]:
# PAT_ref: 0 = no Raman, 1 = Raman recorded
print('PAT_ref distribution:')
print(df[PAT_COL].value_counts())
print(f'Rows with Raman recorded: {(df[PAT_COL] == 1).mean()*100:.1f}%')

In [ ]:
df_raman    = df[df[PAT_COL] == 1].copy()
raman_data  = df_raman[raman_cols].astype(float)
wavenumbers = np.array([int(c) for c in raman_cols])

print(f'Raman data shape: {raman_data.shape}')

mean_spectrum = raman_data.mean(axis=0).values
std_spectrum  = raman_data.std(axis=0).values

all_fault_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values
all_no_fault_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values

raman_fault    = df_raman[df_raman[BATCH_COL].isin(all_fault_ids)][raman_cols].astype(float).mean()
raman_no_fault = df_raman[df_raman[BATCH_COL].isin(all_no_fault_ids)][raman_cols].astype(float).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(wavenumbers, mean_spectrum, color='steelblue', linewidth=0.9)
axes[0].fill_between(wavenumbers, mean_spectrum - std_spectrum, mean_spectrum + std_spectrum,
                     alpha=0.25, color='steelblue')
axes[0].set_xlabel('Wavenumber (cm\u207b\u00b9)')
axes[0].set_ylabel('Intensity')
axes[0].set_title('Mean Raman Spectrum \u00b1 1 Std')

axes[1].plot(wavenumbers, raman_no_fault.values, color='#4C9BE8', linewidth=0.9, label='No-Fault')
axes[1].plot(wavenumbers, raman_fault.values,    color='#E8644C', linewidth=0.9, label='Fault', linestyle='--')
axes[1].set_xlabel('Wavenumber (cm\u207b\u00b9)')
axes[1].set_ylabel('Intensity')
axes[1].set_title('Raman Spectrum \u2014 Fault vs No-Fault Mean Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
raman_var = raman_data.var(axis=0)
raman_var.index = wavenumbers
top20 = raman_var.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(top20.index.astype(str), top20.values, color='steelblue')
ax.set_xlabel('Wavenumber (cm\u207b\u00b9)')
ax.set_ylabel('Variance')
ax.set_title('Raman Spectroscopy \u2014 Top 20 High-Variance Channels')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Missing Value Analysis

In [ ]:
missing = df[process_cols].isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

print(f'Process variables with missing values: {len(missing_nonzero)} / {len(process_cols)}')
print()
print(missing_nonzero.to_string())

if len(missing_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, 3))
    missing_nonzero.plot(kind='bar', ax=ax, color='#E8644C')
    ax.set_ylabel('Missing Ratio')
    ax.set_title('Missing Value Ratio in Process Variables')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Offline variables are measured intermittently
offline_vars = [c for c in process_cols if 'offline' in c.lower() or 'Offline' in c]
print('Offline variables:', offline_vars)

batch1 = df[df[BATCH_COL] == 1]
for v in offline_vars[:3]:
    measured = batch1[v].notna().sum()
    print(f'  {v.split("(")[0].strip()}: measured at {measured}/{len(batch1)} timesteps')

## 7. Fault vs Normal Batch Comparison

In [ ]:
all_fault_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values
all_no_fault_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values

print('Fault batch IDs:', sorted(all_fault_ids))

df_fault    = df[df[BATCH_COL].isin(all_fault_ids)]
df_no_fault = df[df[BATCH_COL].isin(all_no_fault_ids)]

compare_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, var in zip(axes, compare_vars):
    for group, label, color in [
        (df_no_fault, 'No-Fault', '#4C9BE8'),
        (df_fault,    'Fault',    '#E8644C')
    ]:
        mean_t = group.groupby(TIME_COL)[var].mean()
        std_t  = group.groupby(TIME_COL)[var].std()
        ax.plot(mean_t.index, mean_t.values, color=color, linewidth=1.2, label=label)
        ax.fill_between(mean_t.index, mean_t - std_t, mean_t + std_t, alpha=0.15, color=color)
    ax.set_title(var.split('(')[0].strip(), fontsize=9)
    ax.set_xlabel('Time (h)', fontsize=8)
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=9)
fig.suptitle('Fault vs No-Fault \u2014 Mean Time-Series Comparison', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
numeric_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
    'Substrate concentration(S:g/L)',
    'Vessel Volume(V:L)',
    'Aeration rate(Fg:L/h)',
    'Agitator RPM(RPM:RPM)',
    'carbon dioxide percent in off-gas(CO2outgas:%)',
    'Oxygen Uptake Rate(OUR:(g min^{-1}))',
    'Generated heat(Q:kJ)',
]

corr = df_no_fault[numeric_vars].corr()
short_names = [c.split('(')[0].strip() for c in numeric_vars]
corr.index   = short_names
corr.columns = short_names

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Process Variable Correlation (No-Fault Batches)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Summary & Research Direction

In [ ]:
print('=' * 55)
print('EDA Summary')
print('=' * 55)
print(f'  Total batches        : 100 (Fault: 10, Normal: 90)')
print(f'  Total rows           : {len(df):,}')
print(f'  Process variables    : {len(process_cols)}')
print(f'  Raman channels       : {len(raman_cols)} (201~2400 cm\u207b\u00b9)')
print(f'  Raman recorded ratio : {(df[PAT_COL]==1).mean()*100:.1f}%')
print(f'  Yield mean           : {df_stats[YIELD_COL].mean():,.0f} kg')
print(f'  Yield std            : {df_stats[YIELD_COL].std():,.0f} kg')
print('=' * 55)

### Observations (fill in after running EDA)

- [ ] Which variables show the largest difference between Fault and Normal batches?
- [ ] Which Raman channels show high variance, and what compounds might they correspond to?
- [ ] How does the intermittent nature of offline measurements affect the research?
- [ ] Are there differences in timestep lengths across batches? If so, is a batch alignment strategy needed?

### Candidate Research Topics

- **Topic A**: Raman spectroscopy-based soft sensor for real-time penicillin concentration estimation
- **Topic B**: Deep learning-based anomaly detection — LSTM-AE, Transformer-AE
- **Topic C**: Multimodal fusion (process variables + Raman) for batch yield prediction
- **Topic D**: Batch process monitoring — MSPC (PCA-based) vs deep learning comparison

> Finalize research topic based on EDA findings.